# Phân tích hội tụ của Robust Risk-Aware Bandit
Notebook này nạp dữ liệu từ các file `.npz` và vẽ đồ thị so sánh các thuật toán **theo từng alpha**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import glob
import re
from collections import defaultdict

# ─────────────────────────────────────────────
# 1. CẤU HÌNH
# ─────────────────────────────────────────────
results_root = "results"

# Các giá trị N muốn plot (theo thứ tự tăng dần)
n_list = [100, 500, 1000, 5000, 10000, 20000, 50000]

# Các alpha muốn plot — để None thì tự detect từ file
ALPHAS_TO_PLOT = None   # hoặc ví dụ: [0.05, 0.1, 0.2]

# Các function type muốn plot — để None thì plot tất cả
FUNCS_TO_PLOT = None    # hoặc ví dụ: ['linear', 'quadratic']

# Màu và nhãn cho từng thuật toán
color_map = {
    'robust':       '#ff7f0e',   # Cam
    'quantile_risk':'#1f77b4',   # Xanh dương
    'risk_lin_lcb': '#2ca02c',   # Xanh lá
    'risk_exact':   '#d62728',   # Đỏ
        'pessimistic_cdf':  '#9467bd',   # Tím — paper Wan, Li, Wu (2026)
}

label_map = {
    'risk_exact':    'RiskExact NeuralLCB',
    'risk_lin_lcb':  'RiskLin LCB',
    'robust':        'Robust NeuralLCB',
    'quantile_risk': 'Quantile Risk (DDE)',
        'pessimistic_cdf':  'Pessimistic CDF (2605.15620)',
}

# Marker style
marker_map = {
    'robust':       'o',
    'quantile_risk':'s',
    'risk_lin_lcb': '^',
    'risk_exact':   'D',
        'pessimistic_cdf':  'P',   # plus-filled
}

print('Cấu hình xong.')

In [ ]:
# ─────────────────────────────────────────────
# 2. HÀM TIỆN ÍCH
# ─────────────────────────────────────────────

def setup_neurips_style():
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Times New Roman"],
        "axes.labelsize": 13,
        "font.size": 13,
        "legend.fontsize": 9,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.titlesize": 14,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 100
    })

setup_neurips_style()


def get_algo_prefix(fname):
    """Trích xuất prefix thuật toán từ tên file."""
    m = re.match(r'^(.*?)_(robust_syn|mushroom|simglucose)_', fname)
    return m.group(1) if m else None


def get_alpha_from_fname(fname):
    """Trích xuất giá trị alpha từ tên file."""
    m = re.search(r'alpha=([0-9.]+)', fname)
    return float(m.group(1)) if m else None


def get_n_from_fname(fname):
    """Trích xuất N từ tên file."""
    m = re.search(r'_n=(\d+)[_.]', fname)
    return int(m.group(1)) if m else None


def load_one_file(fpath):
    """Load một file npz và trả về dict metrics."""
    d = np.load(fpath, allow_pickle=True)
    algo_name = str(d['algo_names'][0]).replace(' ', '_') if 'algo_names' in d else None

    def get(suffix):
        if algo_name and f"{algo_name}_{suffix}" in d:
            return d[f"{algo_name}_{suffix}"]
        return d[suffix] if suffix in d else None

    out = {}
    regrets = get('regrets')
    if regrets is not None:
        out['regret_mean'] = float(np.mean(regrets))
        out['regret_sem']  = float(np.std(regrets) / np.sqrt(len(regrets.flatten())))

    gt_cvars = get('gt_cvars')
    if gt_cvars is not None:
        oracle_cvar = float(np.mean(d['oracle_cvars'])) if 'oracle_cvars' in d else 0.0
        subopt = oracle_cvar - gt_cvars
        out['gt_cvar_mean'] = float(np.mean(gt_cvars))
        out['gt_cvar_sem']  = float(np.std(gt_cvars) / np.sqrt(len(gt_cvars.flatten())))
        out['subopt_mean']  = float(np.mean(subopt))
        out['subopt_sem']   = float(np.std(subopt) / np.sqrt(len(gt_cvars.flatten())))
        out['oracle_cvar']  = oracle_cvar

    return out


def load_all_data(data_dir, n_list, alpha_filter=None):
    """
    Nạp dữ liệu từ một thư mục kết quả.
    Trả về: dict  [alpha][prefix][n] -> metrics
    """
    all_files = [f for f in glob.glob(os.path.join(data_dir, '*.npz'))
                 if not f.endswith(':Zone.Identifier')]

    result = defaultdict(lambda: defaultdict(dict))  # [alpha][prefix][n]

    for fpath in all_files:
        fname = os.path.basename(fpath)
        prefix = get_algo_prefix(fname)
        alpha  = get_alpha_from_fname(fname)
        n      = get_n_from_fname(fname)
        if prefix is None or alpha is None or n is None:
            continue
        if alpha_filter is not None and alpha not in alpha_filter:
            continue
        if n not in n_list:
            continue
        metrics = load_one_file(fpath)
        if metrics:
            result[alpha][prefix][n] = metrics

    return result


print('Hàm tiện ích đã định nghĩa.')

In [ ]:
# ─────────────────────────────────────────────
# 3. NẠP DỮ LIỆU
# ─────────────────────────────────────────────

alpha_filter = set(ALPHAS_TO_PLOT) if ALPHAS_TO_PLOT else None

all_function_data = {}   # func_name -> {alpha -> {prefix -> {n -> metrics}}}

for dname in sorted(os.listdir(results_root)):
    dpath = os.path.join(results_root, dname)
    if not os.path.isdir(dpath):
        continue
    if not dname.startswith('robust_syn_'):
        continue
    parts = dname.split('_')
    if len(parts) < 3:
        continue
    func_name = parts[2]
    if FUNCS_TO_PLOT is not None and func_name not in FUNCS_TO_PLOT:
        continue
    print(f'Loading: {func_name}  ({dname})')
    all_function_data[func_name] = load_all_data(dpath, n_list, alpha_filter)

print()
for func, d in all_function_data.items():
    alphas = sorted(d.keys())
    print(f'  {func}: alphas = {alphas}')
    for alpha in alphas:
        prefixes = sorted(d[alpha].keys())
        print(f'    alpha={alpha}: algos = {prefixes}')

print('\nDữ liệu đã nạp xong!')

In [ ]:
# ─────────────────────────────────────────────
# 4. HÀM VẼ ĐỒ THỊ THEO ALPHA
# ─────────────────────────────────────────────

def plot_metric_vs_n(ax, alpha_data, n_list, metric_mean, metric_sem,
                     ylabel, title, show_legend=False):
    """
    Vẽ một metric vs N trên ax.
    alpha_data: {prefix -> {n -> metrics}}
    """
    plotted_any = False
    for prefix in sorted(alpha_data.keys()):
        ns, means, sems = [], [], []
        for n in sorted(n_list):
            m = alpha_data[prefix].get(n)
            if m and metric_mean in m:
                ns.append(n)
                means.append(m[metric_mean])
                sems.append(m.get(metric_sem, 0.0))
        if not ns:
            continue
        ns    = np.array(ns)
        means = np.array(means)
        sems  = np.array(sems)
        color  = color_map.get(prefix, '#888888')
        label  = label_map.get(prefix, prefix)
        marker = marker_map.get(prefix, 'o')
        ax.plot(ns, means, color=color, label=label, marker=marker)
        ax.fill_between(ns, means - sems, means + sems, alpha=0.15, color=color)
        plotted_any = True

    ax.set_xscale('log')
    ax.set_xlabel('N (samples)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if show_legend and plotted_any:
        ax.legend(loc='best')


def plot_func_all_alphas(func_name, func_data, save_dir=None):
    """
    Vẽ toàn bộ đồ thị cho một function type.
    Layout: rows = alpha, cols = [CVaR suboptimality, Regret, GT-CVaR]
    """
    alphas = sorted(func_data.keys())
    if not alphas:
        print(f'Không có dữ liệu cho {func_name}')
        return

    METRICS = [
        ('subopt_mean',  'subopt_sem',  'CVaR Suboptimality',   'CVaR Sub-opt ↓'),
        ('regret_mean',  'regret_sem',  'Regret',               'Mean Regret ↓'),
        ('gt_cvar_mean', 'gt_cvar_sem', 'Achieved CVaR (↑=better)', 'Achieved CVaR ↑'),
    ]

    n_rows = len(alphas)
    n_cols = len(METRICS)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5 * n_cols, 3.8 * n_rows),
                             squeeze=False)
    fig.suptitle(f'Function type: {func_name}', fontsize=16, fontweight='bold', y=1.01)

    for row_idx, alpha in enumerate(alphas):
        alpha_data = func_data[alpha]
        for col_idx, (m_mean, m_sem, col_title, ylabel) in enumerate(METRICS):
            ax = axes[row_idx][col_idx]
            show_legend = (col_idx == 0)   # chỉ legend ở cột đầu tiên
            title = f'α={alpha}  |  {col_title}'
            plot_metric_vs_n(ax, alpha_data, n_list, m_mean, m_sem,
                             ylabel=ylabel, title=title, show_legend=show_legend)

    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fpath = os.path.join(save_dir, f'convergence_{func_name}.pdf')
        plt.savefig(fpath, bbox_inches='tight')
        print(f'Saved: {fpath}')
    plt.show()


print('Hàm vẽ đã định nghĩa.')

In [ ]:
# ─────────────────────────────────────────────
# 5. VẼ: MỖI FUNCTION TYPE, TẤT CẢ ALPHA
# Layout: rows = alpha, cols = metric
# ─────────────────────────────────────────────

SAVE_DIR = None   # đặt thành 'figures/' nếu muốn lưu PDF

for func_name in sorted(all_function_data.keys()):
    print(f'\n=== {func_name} ===')
    plot_func_all_alphas(func_name, all_function_data[func_name], save_dir=SAVE_DIR)

In [ ]:
# ─────────────────────────────────────────────
# 6. SO SÁNH THUẬT TOÁN THEO ALPHA (compact)
# Mỗi figure = 1 function, mỗi subplot = 1 metric,
# mỗi line = 1 algo, màu line = alpha
# ─────────────────────────────────────────────

ALPHA_COLORS = {
    0.01: '#003f5c',
    0.05: '#58508d',
    0.1:  '#bc5090',
    0.15: '#ff6361',
    0.2:  '#ffa600',
}
ALPHA_LINESTYLES = {
    0.01: '-',
    0.05: '--',
    0.1:  '-.',
    0.15: ':',
    0.2:  (0, (3, 1, 1, 1)),
}


def plot_algo_alpha_comparison(func_name, func_data, metric_mean, metric_sem,
                               ylabel, save_dir=None):
    """
    1 figure: mỗi subplot = 1 thuật toán.
    Trong mỗi subplot: mỗi line = 1 alpha.
    Metric: metric_mean ± metric_sem vs N.
    """
    alphas  = sorted(func_data.keys())
    # Lấy tập hợp prefix từ tất cả alpha
    prefixes = sorted(set(
        p for a in alphas for p in func_data[a].keys()
    ))
    if not prefixes:
        return

    fig, axes = plt.subplots(1, len(prefixes),
                             figsize=(5 * len(prefixes), 4),
                             squeeze=False)
    fig.suptitle(f'{func_name}  –  {ylabel}', fontsize=15, fontweight='bold')

    for col, prefix in enumerate(prefixes):
        ax = axes[0][col]
        for alpha in alphas:
            d_alpha = func_data[alpha].get(prefix, {})
            ns, means, sems = [], [], []
            for n in sorted(n_list):
                m = d_alpha.get(n)
                if m and metric_mean in m:
                    ns.append(n)
                    means.append(m[metric_mean])
                    sems.append(m.get(metric_sem, 0.0))
            if not ns:
                continue
            ns    = np.array(ns)
            means = np.array(means)
            sems  = np.array(sems)
            c  = ALPHA_COLORS.get(alpha, '#888888')
            ls = ALPHA_LINESTYLES.get(alpha, '-')
            ax.plot(ns, means, color=c, linestyle=ls, label=f'α={alpha}', marker='o', markersize=5)
            ax.fill_between(ns, means - sems, means + sems, alpha=0.12, color=c)

        ax.set_xscale('log')
        ax.set_xlabel('N')
        ax.set_ylabel(ylabel if col == 0 else '')
        ax.set_title(label_map.get(prefix, prefix))
        ax.legend(fontsize=8)

    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_metric = metric_mean.replace('_mean', '')
        fpath = os.path.join(save_dir, f'alpha_compare_{func_name}_{safe_metric}.pdf')
        plt.savefig(fpath, bbox_inches='tight')
        print(f'Saved: {fpath}')
    plt.show()


for func_name in sorted(all_function_data.keys()):
    print(f'\n=== {func_name}: CVaR Suboptimality vs Alpha ===')
    plot_algo_alpha_comparison(func_name, all_function_data[func_name],
                               'subopt_mean', 'subopt_sem',
                               'CVaR Suboptimality ↓', save_dir=SAVE_DIR)
    print(f'=== {func_name}: Regret vs Alpha ===')
    plot_algo_alpha_comparison(func_name, all_function_data[func_name],
                               'regret_mean', 'regret_sem',
                               'Regret ↓', save_dir=SAVE_DIR)

In [ ]:
# ─────────────────────────────────────────────
# 7. BẢNG TỔNG HỢP: suboptimality tại N lớn nhất
# ─────────────────────────────────────────────

N_SUMMARY = max(n_list)   # lấy N lớn nhất có sẵn

for func_name in sorted(all_function_data.keys()):
    print(f'\n=== {func_name}  |  CVaR Suboptimality @ N={N_SUMMARY} ===')
    print(f'  {"Alpha":>8} | ' + ' | '.join(f'{label_map.get(p, p):>28}'
          for p in sorted(color_map.keys())))
    print('  ' + '-' * (12 + 32 * len(color_map)))

    for alpha in sorted(all_function_data[func_name].keys()):
        row = f'  {alpha:>8} |'
        for prefix in sorted(color_map.keys()):
            m = all_function_data[func_name][alpha].get(prefix, {}).get(N_SUMMARY)
            if m and 'subopt_mean' in m:
                val = f"{m['subopt_mean']:.4f} ± {m['subopt_sem']:.4f}"
            else:
                val = 'N/A'
            row += f' {val:>28} |'
        print(row)